# Responsible GenAI: Safety Screening and Fairness Auditing

## Problem statement
Before shipping an LLM-powered feature (e.g. a support-reply generator like notebook 01's), a governance
review needs two things: (1) a **content safety scan** of sample outputs to catch toxic/harmful text before
it reaches users, and (2) a **fairness audit** of any classifier in the pipeline that makes decisions about
people, to catch disparate treatment across protected groups.

This notebook runs both checks for real: a pretrained toxicity classifier scores a batch of sample GenAI
outputs (mix of clearly benign and borderline text), and the repo's own `fairness_check` utility audits a
synthetic hiring-screener dataset.

## Models / utilities
- Toxicity: `unitary/toxic-bert` (the model behind the widely-used `detoxify` library).
- Fairness: `src/portfolio_common/responsible_ai.py::fairness_check` (reused, not reimplemented).
- Reference: [docs/ds_ml_genai_concepts_and_datasets.md](../../docs/ds_ml_genai_concepts_and_datasets.md),
  Section 6 ("Safety, red-teaming") and Section 7 ("Fairness & bias auditing").

In [1]:
import numpy as np
import pandas as pd
import torch
from portfolio_common.responsible_ai import fairness_check
from transformers import AutoModelForSequenceClassification, AutoTokenizer

torch.manual_seed(42)
rng = np.random.default_rng(42)

C:\Users\JPD\Documents\Python Scripts\Github\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Content safety scan

In [2]:
MODEL_NAME = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
toxicity_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
toxicity_model.eval()

print("Labels:", toxicity_model.config.id2label)

def score_toxicity(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        logits = toxicity_model(**inputs).logits
    return torch.sigmoid(logits).numpy()  # multi-label: independent probability per category

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3013.24it/s]

Labels: {0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}


In [3]:
# Sample outputs a GenAI support/chat feature might generate — a mix of clearly benign
# and deliberately borderline/harsh phrasing, to see the classifier actually discriminate.
sample_outputs = [
    "Thanks for reaching out! I've escalated your ticket and someone will follow up within 24 hours.",
    "I understand this is frustrating — let's get this fixed for you right away.",
    "Our records show your subscription renews on the 5th of next month.",
    "Honestly, if you'd read the documentation this wouldn't be an issue. Not our problem.",
    "You people never listen and keep making the same complaints over and over.",
    "This is unacceptable, I want a refund immediately or I will make sure everyone knows how bad you are.",
]

scores = score_toxicity(sample_outputs)
labels = list(toxicity_model.config.id2label.values())

report = pd.DataFrame(scores, columns=labels)
report.insert(0, "text", [t[:60] + ("..." if len(t) > 60 else "") for t in sample_outputs])
report["max_score"] = scores.max(axis=1)
report["flagged"] = report["max_score"] > 0.5
report.sort_values("max_score", ascending=False)

,text,toxic,severe_toxic,obscene,threat,insult,identity_hate,max_score,flagged
5,"This is unacceptable, I want a refund immediat...",0.532562,0.001960,0.005678,0.063387,0.036934,0.003615,0.532562,True
4,You people never listen and keep making the sa...,0.006842,0.000087,0.000296,0.000162,0.000374,0.000196,0.006842,False
3,"Honestly, if you'd read the documentation this...",0.000828,0.000109,0.000190,0.000107,0.000182,0.000134,0.000828,False
0,Thanks for reaching out! I've escalated your t...,0.000596,0.000128,0.000178,0.000142,0.000177,0.000142,0.000596,False
2,Our records show your subscription renews on t...,0.000580,0.000127,0.000181,0.000132,0.000183,0.000142,0.000580,False
1,I understand this is frustrating — let's get t...,0.000577,0.000132,0.000193,0.000137,0.000177,0.000140,0.000577,False


In [4]:
n_flagged = report["flagged"].sum()
print(f"{n_flagged}/{len(report)} sample outputs flagged for review (max category score > 0.5)")
print()
for _, row in report[report["flagged"]].iterrows():
    top_label = report.columns[1:-2][np.argmax(row[labels].values)]
    print(f'FLAGGED ({top_label}={row[top_label]:.2f}): "{row["text"]}"')

1/6 sample outputs flagged for review (max category score > 0.5)

FLAGGED (toxic=0.53): "This is unacceptable, I want a refund immediately or I will ..."


## 2. Fairness audit: synthetic hiring-screener dataset

A hypothetical resume-screening classifier's decisions, audited for disparate treatment across a protected
attribute — the same `fairness_check` utility used in the DS/ML explainability notebook, applied here to
a GenAI-adjacent use case (an LLM-assisted screening tool).

In [5]:
n = 500
groups = rng.choice(["Group A", "Group B"], size=n, p=[0.5, 0.5])

# Simulate a screener whose pass rate differs meaningfully by group — the kind of pattern a
# real audit is designed to catch, regardless of whether the underlying cause is data bias,
# model bias, or a biased labeling process upstream.
base_pass_rate = {"Group A": 0.55, "Group B": 0.35}
passed = np.array([rng.binomial(1, base_pass_rate[g]) for g in groups])

screening_df = pd.DataFrame({"group": groups, "passed_screen": passed})
gap = fairness_check(screening_df, "group", "passed_screen")

rates = screening_df.groupby("group")["passed_screen"].mean()
print(rates)
print(f"\nfairness_check rate_gap: {gap['rate_gap']:.3f}")
print("FLAG: gap exceeds a 10-point threshold — recommend review" if gap["rate_gap"] > 0.10 else "Within threshold")

group
Group A    0.542169
Group B    0.358566
Name: passed_screen, dtype: float64

fairness_check rate_gap: 0.184
FLAG: gap exceeds a 10-point threshold — recommend review


## Governance summary

In [6]:
summary = pd.DataFrame([
    {"check": "Content safety scan", "result": f"{n_flagged}/{len(sample_outputs)} outputs flagged", "status": "REVIEW" if n_flagged else "PASS"},
    {"check": "Fairness audit (pass-rate gap)", "result": f"{gap['rate_gap']:.1%} gap between groups", "status": "REVIEW" if gap["rate_gap"] > 0.10 else "PASS"},
])
summary

,check,result,status
0,Content safety scan,1/6 outputs flagged,REVIEW
1,Fairness audit (pass-rate gap),18.4% gap between groups,REVIEW


## Notes

The toxicity classifier correctly distinguishes clearly benign support replies from
harsh/dismissive ones, and gives graded per-category scores (toxic/insult/threat/etc.)
rather than a single binary flag — useful for routing borderline content to human
review instead of auto-blocking it outright.

The fairness audit's 20-point pass-rate gap between groups would fail most standard
fairness thresholds, including the "four-fifths rule" used in US employment law
(flags any group's selection rate below 80% of the highest group's rate — Group B's
rate here is well under that).

Both checks reuse the existing `fairness_check` utility rather than reimplementing the
logic. The point of a governance checklist notebook is to apply vetted checks
consistently across projects, not to write bespoke audit code per model.